# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: KMeans clustering.** My lane is Structured Content Archetype Clustering — there's no
correct label to classify against, so a supervised method (logistic regression, decision tree)
would be answering a question I haven't defined a target for. Clustering fits because the goal is
to discover natural groupings in content behavior, not predict a known outcome.

To make the result checkable against something real, I evaluate the clusters against
trend_direction — a genuine observed future-ish signal I never feed into the clustering features
themselves — using the same precision@K yardstick my Week-4 baseline uses.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
# find the repo root from wherever this kernel started (same pattern as w04)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Rows: {len(df)}, clients: {df['client_id'].nunique()}")

Rows: 30000, clients: 32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client_id, 75/25.** Rows from the same client share hidden characteristics —
their industry, their content style, their baseline traffic level. A random row-level split would
let the model see 3 of a client's 10 pages in training and quietly "recognize" that client's
pattern when scoring the other 7 in test — that's memorization, not generalization. I verified
zero client overlap between my train and test sets below.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print(f"train rows: {len(train)}, test rows: {len(test)}")
print(f"train clients: {train['client_id'].nunique()}, test clients: {test['client_id'].nunique()}")
print(f"client overlap between train/test: {len(set(train['client_id']) & set(test['client_id']))}")

train rows: 22885, test rows: 7115
train clients: 24, test clients: 8
client overlap between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Two columns had real missing values (word_count: 7,699 missing, scroll_rate: 125 missing) —
# flagging before filling, not silently zeroing them out
for col in ['word_count', 'scroll_rate']:
    df[f'{col}_was_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())

train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
train['is_declining'] = (train['trend_direction'] == 'down').astype(int)
test['is_declining'] = (test['trend_direction'] == 'down').astype(int)

feat_cols = ['avg_position', 'ctr', 'impressions_90d', 'word_count', 'content_age_days', 'engagement_rate', 'scroll_rate']
scaler = StandardScaler().fit(train[feat_cols])
Xtr, Xte = scaler.transform(train[feat_cols]), scaler.transform(test[feat_cols])

km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xtr)
train['cluster'], test['cluster'] = km.labels_, km.predict(Xte)

sil = silhouette_score(Xtr, train['cluster'], sample_size=5000, random_state=42)
print(f"Silhouette (train, sampled): {sil:.3f}")

# pick the riskiest archetype using TRAIN outcomes only — never peek at test here
risk_by_cluster = train.groupby('cluster')['is_declining'].agg(['mean', 'count'])
print(risk_by_cluster)
risky_cluster = risk_by_cluster['mean'].idxmax()
print(f"Riskiest cluster: {risky_cluster}")

# --- baseline, computed on TEST only, same K ---
K = 200
peer_avg_ctr = test.groupby('position_tier')['ctr'].transform('mean')
visible = (test['impressions_90d'] >= 300).astype(int)
ctr_gap = (peer_avg_ctr - test['ctr']).clip(lower=0)
test['baseline_action_score'] = visible * ctr_gap * np.log1p(test['impressions_90d'])
baseline_topk = test.sort_values('baseline_action_score', ascending=False).head(K)
baseline_precision = baseline_topk['is_declining'].mean()

# --- cluster-based ranking: within the risky cluster, closest-to-centroid first ---
centroid = km.cluster_centers_[risky_cluster]
test_c = test[test['cluster'] == risky_cluster].copy()
test_c['dist_to_centroid'] = np.linalg.norm(scaler.transform(test_c[feat_cols]) - centroid, axis=1)
cluster_topk = test_c.sort_values('dist_to_centroid').head(K)
cluster_precision = cluster_topk['is_declining'].mean()

base_rate = test['is_declining'].mean()

comparison = pd.DataFrame({
    "method": ["base rate (random)", "Week-4 baseline rule", "Week-5 cluster archetype"],
    f"precision@{K}": [base_rate, baseline_precision, cluster_precision],
})
comparison

Silhouette (train, sampled): 0.279
             mean  count
cluster                 
0        0.633934   3330
1        0.634873   9268
2        0.438319   8090
3        0.497175    354
4        0.472056   1843
Riskiest cluster: 1


,method,precision@200
0,base rate (random),0.516514
1,Week-4 baseline rule,0.730000
2,Week-5 cluster archetype,0.590000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**The baseline still wins.** My Week-4 rule scored 0.730 precision@200 against real future decline,
versus 0.590 for the best cluster archetype — both clear improvements over the 0.517 base rate, but
the hand-written rule is still doing more work. That's a legitimate result, not a failure: the
CTR-vs-position signal I confirmed in Week 4 is a sharp, targeted trigger, while clustering optimizes
for grouping pages that look similar overall — it isn't specifically optimized to separate
"declining" from "not declining," so losing to a rule built exactly for that job makes sense.

**What each archetype actually looks like** (from train-set feature averages):
- Cluster 1 (riskiest, 9,268 pages): moderate traffic (~4,330 impressions), weak CTR (0.30), low
  engagement (1.96%) and low scroll depth — pages getting seen but not read. 63% later declined.
- Cluster 0 (nearly as risky, 3,330 pages): much higher traffic (~19,276 impressions) but similarly
  weak CTR (0.26) and worse average position (20.7) — big, established pages quietly losing ground.
- Cluster 2 (8,090 pages): stronger CTR (0.88), older content (384 days), lowest decline rate (44%)
  — steady performers.
- Clusters 3 and 4 (small, 354 and 1,843 pages): very high engagement/scroll rates but tiny traffic
  — niche pages, decline rate close to the base rate either way.

**Takeaway:** clustering surfaced a real, interpretable pattern (moderate-traffic, low-CTR pages skew
toward decline), but it's a blunter instrument than a rule aimed directly at the CTR-position gap.
For the capstone, I'd keep clustering for archetype labeling and reporting, but keep the sharper
CTR-fix rule as the actual prioritization mechanism.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.